# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

For NNs there are 2 "main" complexity interpretations:
- depth: number of hidden layers
- width: number of nodes per layer
- (more complex activation function) - maybe as an extra
- (more complex optimizer) - maybe as an extra

So we decide to test 3 different NN structures:

__baseline model__:
- hidden layers: 2
- nodes per layer: 64

__wider model__:
- hidden layers: 2
- nodes per layer: 128

__deeper model__:
- hidden layers: 4
- nodes per layer: 64

For better comparison, we will test all three architectures with the same shared configurations.

In [35]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
# import matplotlib.pyplot as plt

# modeling
import copy
import random
import types

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# reset working dir
import os
from pathlib import Path


In [36]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data= pd.read_csv("data/aggregated/hexagon/demand_hex_1h_high.csv")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Shared Configs                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# --- optimization ---
OPTIMIZER          = "adam"
LEARNING_RATE      = 1e-4
BATCH_SIZE         = 256
MAX_EPOCHS         = 100

# --- loss / output ---
LOSS               = "poisson_nll"     # demand is count data
OUTPUT_UNITS       = 1
OUTPUT_ACTIVATION  = "softplus"        # non-negative expected count

# --- layer defaults ---
HIDDEN_ACTIVATION  = "relu"
WEIGHT_INIT        = "he_normal"

# TODO: remove if not needed, depends on the degree of overfitting
# --- regularization (off by default to isolate the complexity effect) ---
DROPOUT            = 0.0
WEIGHT_DECAY       = 0.0

# --- early stopping ---
EARLY_STOPPING     = True
MONITOR            = "val_loss"
PATIENCE           = 10

# --- data handling ---
SPLIT              = "temporal"        # earlier 2025 -> train, later -> val/test
SCALER_FIT_ON      = "train_only"
SHUFFLE_TIME       = False             # never shuffle across time (no leakage)

# --- reproducibility ---
SEEDS              = (0, 1, 2, 3, 4)   # run each model across all seeds; report mean +/- std

# --- architectures (the ONLY thing that varies across the 3 models) ---
# format: (n_hidden_layers, width)
ARCH_BASELINE      = (2, 64)
ARCH_WIDER         = (2, 128)          # depth fixed, width up
ARCH_DEEPER        = (4, 64)           # width fixed, depth up
# expose as module-like object so model code can use config.XXX
config = types.SimpleNamespace(
    LEARNING_RATE  = LEARNING_RATE,
    WEIGHT_DECAY   = WEIGHT_DECAY,
    BATCH_SIZE     = BATCH_SIZE,
    MAX_EPOCHS     = MAX_EPOCHS,
    EARLY_STOPPING = EARLY_STOPPING,
    PATIENCE       = PATIENCE,
    OUTPUT_UNITS   = OUTPUT_UNITS,
)


In [32]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #
1
class DemandBaseline(nn.Module):
    def __init__(self, input_dim, n_layers, width):
        super().__init__()
        layers = []
        in_dim = input_dim # tracks input size of next layer/number of outputs of current layer going into next layer
        for _ in range(n_layers):
            linear = nn.Linear(in_dim, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")  # He initialization for ReLU, he/kaiming init keeps variance of weights constant
            nn.init.zeros_(linear.bias) # biases start at 0
            layers += [linear, nn.ReLU()] # hidden layers with ReLU activation
            in_dim = width
        out = nn.Linear(in_dim, config.OUTPUT_UNITS) # output layer maps to 1 unit
        nn.init.kaiming_normal_(out.weight, nonlinearity="relu")
        nn.init.zeros_(out.bias) # output layer bias starts at 0
        layers += [out, nn.Softplus()]   # forces the output to be non-negative
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1) # remove shape so only prediction is left


# --------------------------------------------------------------------------- #
# 2. Reproducibility
# --------------------------------------------------------------------------- #
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- #
# 3. Training loop (Adam + early stopping, all values from config)
# --------------------------------------------------------------------------- #
def train_model(arch, X_train, y_train, X_val, y_val, seed=0, device="cpu", verbose=True):
    set_seed(seed)
    n_layers, width = arch
    model = DemandBaseline(X_train.shape[1], n_layers, width).to(device)

    loss_fn = nn.PoissonNLLLoss(log_input=False, full=False)  # input is the rate
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY, # decay is 0 by default, no regularization
    )

    train_dl = DataLoader(
        TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                      torch.as_tensor(y_train, dtype=torch.float32)),
        batch_size=config.BATCH_SIZE,
        shuffle=True,        # rows within the training set may shuffle; the
                             # train/val SPLIT itself stays temporal (no leakage)
    )
    X_val_t = torch.as_tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.as_tensor(y_val, dtype=torch.float32).to(device)

    best_val, best_state, wait = float("inf"), None, 0 # early stopping
    epoch_width = len(str(config.MAX_EPOCHS))

    for epoch in range(config.MAX_EPOCHS):
        # ── train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss, n_batches = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches    += 1
        train_loss = running_loss / n_batches

        # ── validate ───────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        # ── early stopping bookkeeping ─────────────────────────────────────────
        improved = val_loss < best_val
        if improved:
            best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1

        if verbose:
            marker = " *" if improved else f" (no improvement {wait}/{config.PATIENCE})"
            print(f"  epoch {epoch+1:{epoch_width}d}/{config.MAX_EPOCHS}"
                  f"  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if config.EARLY_STOPPING and wait >= config.PATIENCE:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)   # restore best weights
    return model, best_val


In [8]:
data.head()

,pickup_h3_high_resolution,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,882664d98bfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,882664c837fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,8826645005fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,882664d8dbfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,882664ce21fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
# Source - https://stackoverflow.com/a/18145399
# Posted by LondonRob, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-08, License - CC BY-SA 4.0

data = data.drop('hour_stamp_since_epoch', axis=1)


In [7]:
len(data)

17118994

In [9]:
data.columns

Index(['pickup_h3_high_resolution', 'time_bucket', 'trip_seconds',
       'trip_miles', 'fare', 'tips', 'tolls', 'extras', 'trip_total',
       'pickup_centroid_latitude', 'pickup_centroid_longitude',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'index',
       'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m',
       'direct_radiation', 'avg_idle_time', 'rush_hour', 'trip_count',
       'active_taxis', 'start_month', 'start_hour', 'day_of_week',
       'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_2m',
       'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth',
       'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration',
       'start_hour_sin', 'start_hour_cos', 'start_month_sin',
       'start_month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'avg_trip_duration', 'avg_trip_distance', 'avg_fare', 'avg_trip_total',
       'avg_tip', 'tip_rate'],
      dtype='str')

In [ ]:
# n hourly timestamps in 2025 multiplied by number of hexagons = number of rows we expect in data


In [ ]:
n_unique_hexagons = data['pickup_h3_high_resolution'].nunique()
print(f"Number of unique hexagons: {n_unique_hexagons}")

In [23]:
date_only_rows = data[~data['time_bucket'].str.contains(' ')]
print(f"Rows without time component: {len(date_only_rows)}")
date_only_rows[['time_bucket', 'pickup_h3_high_resolution']].drop_duplicates()

Rows without time component: 354


,time_bucket,pickup_h3_high_resolution
17118640,2026-01-01,882664c84dfffff
17118641,2026-01-01,882664c80dfffff
17118642,2026-01-01,882664d9a1fffff
17118643,2026-01-01,8826645765fffff
17118644,2026-01-01,882664cee7fffff
...,...,...
17118989,2026-01-01,882664562dfffff
17118990,2026-01-01,882664c9d5fffff
17118991,2026-01-01,882664c925fffff
17118992,2026-01-01,8826645043fffff


In [21]:
expected_range = pd.date_range(start="2025-01-01", end="2026-01-01", freq="h", inclusive="left")
actual_timestamps = pd.to_datetime(data['time_bucket'], format='mixed').unique()
missing = expected_range.difference(actual_timestamps)
print(f"Expected timestamps : {len(expected_range)}")
print(f"Actual timestamps   : {len(actual_timestamps)}")
print(f"Missing timestamps  : {len(missing)}")
if len(missing) > 0:
    print(missing)

Expected timestamps : 8760
Actual timestamps   : 8761
Missing timestamps  : 0


In [26]:
data.iloc[[17118640]]

,pickup_h3_high_resolution,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
17118640,882664c84dfffff,2026-01-01,1827.0,10.51,28.25,0.0,0.0,0.0,28.25,41.927261,...,0.5,0.866025,0.433884,-0.900969,1827.0,10.51,28.25,28.25,0.0,0.0


In [28]:
print(data["time_bucket"][17118640])

2026-01-01


## Train / Val / Test Split

We split on **unique timestamps** (not rows) so that every hexagon for a given
hour lands in exactly one split — no leakage across the temporal boundary.

`TimeSeriesSplit(n_splits=2, test_size=≈15 %)` produces three contiguous segments:

| Split | Share | Period |
|-------|-------|--------|
| Train | ~70 % | Jan → early Oct |
| Val   | ~15 % | early Oct → mid Nov |
| Test  | ~15 % | mid Nov → Dec |

In [27]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Train / Val / Test Split                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data['time_bucket'] = pd.to_datetime(data['time_bucket'], format='mixed')

timestamps = pd.DatetimeIndex(np.sort(data['time_bucket'].unique()))
n_ts = len(timestamps)

# ~70 / 15 / 15 split on unique timestamps
test_size = int(round(0.15 * n_ts))
tscv = TimeSeriesSplit(n_splits=2, test_size=test_size)
splits = list(tscv.split(timestamps))

# fold 0 → train | val,  fold 1 → (train+val) | test
train_ts = timestamps[splits[0][0]]
val_ts   = timestamps[splits[0][1]]
test_ts  = timestamps[splits[1][1]]

train_data = data[data['time_bucket'].isin(train_ts)].reset_index(drop=True)
val_data   = data[data['time_bucket'].isin(val_ts)].reset_index(drop=True)
test_data  = data[data['time_bucket'].isin(test_ts)].reset_index(drop=True)

print(f"Train : {len(train_ts):5d} timestamps  |  {train_ts[0].date()} → {train_ts[-1].date()}  |  {len(train_data):>9,} rows")
print(f"Val   : {len(val_ts):5d} timestamps  |  {val_ts[0].date()} → {val_ts[-1].date()}  |  {len(val_data):>9,} rows")
print(f"Test  : {len(test_ts):5d} timestamps  |  {test_ts[0].date()} → {test_ts[-1].date()}  |  {len(test_data):>9,} rows")


Train :  6133 timestamps  |  2025-01-01 → 2025-09-13  |  11,983,882 rows
Val   :  1314 timestamps  |  2025-09-13 → 2025-11-07  |  2,567,556 rows
Test  :  1314 timestamps  |  2025-11-07 → 2026-01-01  |  2,567,556 rows


## Feature Preparation

Drop leakage columns (aggregated trip statistics that are derived from the same
time-bucket), ID/index columns, and the target.  
Fit a `StandardScaler` on the **training set only** to avoid leakage into val/test.


In [28]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Feature Preparation                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# leakage: these are computed from the trips in the same bucket
LEAKAGE_COLS = [
    'trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls', 'extras', 'trip_total',
    'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance',
    'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'active_taxis',
    'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
]
ID_COLS    = ['pickup_h3_high_resolution', 'time_bucket']
TARGET_COL = 'trip_count'

FEATURE_COLS = [
    c for c in train_data.columns
    if c not in LEAKAGE_COLS + ID_COLS + [TARGET_COL]
]
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
X_val   = scaler.transform(val_data[FEATURE_COLS].values)
X_test  = scaler.transform(test_data[FEATURE_COLS].values)

y_train = train_data[TARGET_COL].values.astype(float)
y_val   = val_data[TARGET_COL].values.astype(float)
y_test  = test_data[TARGET_COL].values.astype(float)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")


Features (30): ['pickup_centroid_latitude', 'pickup_centroid_longitude', 'index', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m', 'direct_radiation', 'rush_hour', 'start_month', 'start_hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_2m', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration', 'start_hour_sin', 'start_hour_cos', 'start_month_sin', 'start_month_cos', 'day_of_week_sin', 'day_of_week_cos']
X_train : (11983882, 30)   y_train : (11983882,)
X_val   : (2567556, 30)     y_val   : (2567556,)
X_test  : (2567556, 30)    y_test  : (2567556,)


## Baseline Model — Training

Run `ARCH_BASELINE = (2 hidden layers, 64 units)` across all seeds and report
the mean ± std validation loss.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

baseline_val_losses  = []
baseline_models      = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        seed=seed, device=device,
    )
    baseline_val_losses.append(val_loss)
    baseline_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nBaseline  val loss:  {np.mean(baseline_val_losses):.4f} ± {np.std(baseline_val_losses):.4f}")


Device: cpu

  epoch   1/100  train=-0.1073  val=-0.1499 *
  epoch   2/100  train=-0.1503  val=-0.1609 *
  epoch   3/100  train=-0.1609  val=-0.1558 (no improvement 1/10)
  epoch   4/100  train=-0.1662  val=-0.1223 (no improvement 2/10)
  epoch   5/100  train=-0.1658  val=-0.0341 (no improvement 3/10)
  epoch   6/100  train=-0.1680  val=0.0524 (no improvement 4/10)
  epoch   7/100  train=-0.1704  val=-0.0172 (no improvement 5/10)
  epoch   8/100  train=-0.1712  val=-0.0641 (no improvement 6/10)
  epoch   9/100  train=-0.1722  val=-0.0096 (no improvement 7/10)
  epoch  10/100  train=-0.1732  val=0.0137 (no improvement 8/10)
  epoch  11/100  train=-0.1743  val=0.0402 (no improvement 9/10)
  epoch  12/100  train=-0.1755  val=0.0274 (no improvement 10/10)
  early stopping at epoch 12
  seed=0  val_loss=-0.1609


## Baseline Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **MAPE** — mean absolute percentage error (relative, skip zeros)


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

X_test_t = torch.as_tensor(X_test, dtype=torch.float32).to(device)

mae_scores, rmse_scores, mape_scores = [], [], []

for seed, model in zip(SEEDS, baseline_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0                                  # avoid division by zero
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

print(f"\nBaseline  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Baseline  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Baseline  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")
